# MCP Tool Testing Notebook

This notebook gives you a repeatable way to test each MCP tool independently.

## Goal
- Validate input/output contracts per tool
- Measure per-tool latency
- Quickly isolate whether a failure is in tool logic vs graph orchestration

## Prerequisites
1. Activate venv in terminal: `\.venv\Scripts\Activate.ps1`
2. Install Jupyter in venv if needed: `python -m pip install jupyter ipykernel`
3. Start Jupyter from repo root: `python -m jupyter lab`


In [ ]:
from pathlib import Path
import os
import pandas as pd

os.chdir(Path.cwd())
Path.cwd()

In [ ]:
from learning.debug_helpers import base_config, mcp_call, set_neo4j

cfg = base_config()
cfg

## 1) `load_csv` tool
Expected: returns `records`, `row_count`, `invalid_rows`, `warnings`.

In [ ]:
result, ms = mcp_call('load_csv', {'file_path': 'data/inventory_mock.csv'})
print(f'load_csv latency: {ms:.2f} ms')
print('row_count:', result['row_count'])
print('valid records:', len(result['records']))
print('invalid rows:', len(result['invalid_rows']))
result['records'][0]

## 2) `calc_metrics` tool
Expected: one SKU in, one metrics payload out.

In [ ]:
sku = result['records'][0]
metric, ms = mcp_call('calc_metrics', {'sku': sku, 'config': cfg})
print(f'calc_metrics latency: {ms:.2f} ms')
metric

## 3) `fetch_rules` tool
Expected: default/file source and active rule list.

In [ ]:
rules, ms = mcp_call('fetch_rules', {'config_path': 'config/thresholds.yaml', 'category': None})
print(f'fetch_rules latency: {ms:.2f} ms')
rules

## 4) `query_graph` tool (fallback path)
Force Neo4j off to verify NetworkX/cache behavior.

In [ ]:
set_neo4j(False)
ctx1, ms1 = mcp_call('query_graph', {'sku_id': 'SKU-001', 'category': 'electronics', 'query_type': 'all', 'config': cfg})
ctx2, ms2 = mcp_call('query_graph', {'sku_id': 'SKU-001', 'category': 'electronics', 'query_type': 'all', 'config': cfg})
print(f'first call latency: {ms1:.2f} ms | source={ctx1.get("source")}')
print(f'second call latency: {ms2:.2f} ms | source={ctx2.get("source")}')
ctx1, ctx2

## MCP Smoke Matrix
Run this to get a one-shot health table for all tools.

In [ ]:
checks = []
for name, args in [
    ('load_csv', {'file_path': 'data/inventory_mock.csv'}),
    ('fetch_rules', {'config_path': 'config/thresholds.yaml', 'category': None}),
    ('query_graph', {'sku_id': 'SKU-002', 'category': 'electronics', 'query_type': 'all', 'config': cfg}),
]:
    try:
        out, ms = mcp_call(name, args)
        checks.append({'tool': name, 'ok': True, 'latency_ms': round(ms, 2), 'keys': list(out.keys())[:6]})
    except Exception as e:
        checks.append({'tool': name, 'ok': False, 'latency_ms': None, 'error': str(e)})

pd.DataFrame(checks)